In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 🟢 Escenario 1: El Modelo LLM (Sin extras)

En este primer caso, el modelo funciona únicamente con su **memoria de entrenamiento**. No tiene acceso a archivos externos, ni a internet, ni a bases de datos actualizadas.

### Ejemplo A: Conocimiento General
* **Prompt:** "¿Cómo se dice gato en alemán?"
* **Respuesta del LLM:** "Gato en alemán se dice **Katze**."
* **Resultado:** ✅ **Éxito**. El modelo es multilingüe y conoce conceptos universales.

### Ejemplo B: Conocimiento Específico (Nuestro Centro)
* **Prompt:** "¿A qué hora empiezan las clases en el IES Portada Alta?"
* **Respuesta del LLM:** "Las clases en España suelen empezar a las 8:00 o 8:30."
* **Resultado:** ❌ **Alucinación**. Se inventa un dato basado en estadísticas generales.

### Ejemplo C: La Falta de Persistencia (Sin Base de Datos)
Si yo le digo: *"En Portada Alta las clases empiezan a las 8:15"*.
Y luego, en una sesión nueva, le pregunto: * "¿Y a qué hora terminan?"*.
* **Respuesta del LLM:** "No estoy seguro de a qué institución te refieres o de qué horario estás hablando."
* **Resultado:** ❌ **Olvido**. Al no tener una base de datos de contexto donde guardar la información de nuestro instituto de forma permanente, el modelo "olvida" todo en cuanto cerramos el chat. No tiene un "disco duro" propio con nuestra información.

## 🔵 Escenario 2: El Modelo con RAG (Base de Datos del Instituto)

Ahora añadimos una **Base de Datos de Conocimiento** usando **LlamaIndex**. Hemos indexado documentos internos del IES Portada Alta en una base de datos vectorial. El sistema tiene un "Orquestador" que decide de dónde sacar la información.

### Ejemplo A: Consulta Específica (Uso de RAG)
* **Prompt:** "¿A qué hora empiezan las clases en el instituto Portada Alta?"
* **Proceso:** 1. El orquestador busca en la **Base de Datos Vectorial**.
    2. Encuentra el fragmento: *"Horario Portada Alta: 08:15 a 14:45"*.
    3. Le pasa ese dato al LLM.
* **Respuesta del LLM:** "Las clases en Portada Alta empiezan a las **8:15**."
* **Resultado:** ✅ **Precisión total**. Ya no inventa; consulta fuentes fiables.

### Ejemplo B: El problema del Contexto (Sin Memoria)
Si inmediatamente después preguntamos: *"¿Y a qué hora terminan?"*.
* **Respuesta del LLM:** "¿A qué te refieres con 'terminan'? ¿Las clases de qué lugar?"
* **Resultado:** ❌ **Falta de Contexto**. Aunque el RAG tiene la información en el "libro" (Base de Datos de Conocimiento), el sistema ha olvidado de qué estábamos hablando porque no tiene una **Base de Datos de Contexto** (Memoria de sesión) que guarde el hilo de la charla.

### Ejemplo C: Información que NO existe (El peligro)
* **Prompt:** "¿Cuál es el menú de la cafetería?"
* **Resultado posible 1 (Sin control):** **Alucinación**. Se inventa un menú basado en otros institutos.
* **Resultado posible 2 (Con control):** El modelo responde: *"Lo siento, no tengo información sobre el menú en mis documentos"*.

## 🟠 Escenario 3: El Sistema Completo (Gemini / ChatGPT / Web-RAG)

En este nivel, no usamos solo un modelo, sino una **arquitectura de ingeniería completa**. Aquí el RAG no se limita a un PDF, sino que tiene acceso a **Internet** y a una **Base de Datos de Contexto** (Memoria).

### Ejemplo A: Conocimiento General (Entrenamiento)
* **Prompt:** "¿Cuántos países hay en la Unión Europea?"
* **Respuesta:** "Hay 27 países miembros."
* **Resultado:** ✅ Responde directamente desde su memoria interna de entrenamiento.

### Ejemplo B: Conocimiento Actualizado (Web-RAG)
* **Prompt:** "¿A qué hora empiezan las clases en el instituto Portada Alta de Málaga?"
* **Proceso:** 1. El modelo no lo sabe por su entrenamiento.
    2. El "orquestador" lanza una búsqueda en **Google/Bing** (Web-RAG).
    3. Lee la web del instituto y extrae el dato.
* **Respuesta:** "Las clases en Portada Alta empiezan a las **8:15**."
* **Resultado:** ✅ **Éxito**. Usa "todo Internet" como su base de datos de conocimiento.

### Ejemplo C: Memoria Persistente (Base de Datos de Contexto)
* **Prompt:** "¿Y cuándo es el recreo?"
* **Proceso:** El sistema consulta su **Base de Datos de Contexto**. Sabe que seguimos hablando de "Portada Alta". Busca de nuevo en la info recuperada de la web.
* **Respuesta:** "El recreo en Portada Alta es de **11:15 a 11:45**."
* **Resultado:** ✅ **Conversación fluida**. A diferencia del Escenario 2, aquí el sistema sí recuerda de qué estamos hablando.

| Feature | 🟢 Scenario 1 (LLM no extras) | 🔵 Scenario 2 (Local RAG) | 🟠 Scenario 3 (Web-RAG + Memory) |
| :--- | :---: | :---: | :---: |
| **General Knowledge** | ✅ Yes | ✅ Yes | ✅ Yes |
| **Our School Data** | ❌ No (Hallucinates) | ✅ Yes (Via LlamaIndex) | ✅ Yes (Via Google/Web) |
| **Real-Time Info** | ❌ No | ⚠️ Only if we update PDFs | ✅ Yes (Internet Access) |
| **Conversation Thread** | ❌ No | ❌ No (No Context DB) | ✅ Yes (Session Memory) |

# Explicación Técnica: Sistema RAG (Generación Aumentada por Recuperación)

## 1. ¿Qué es un Sistema RAG?
Un sistema **RAG** es una arquitectura que permite a un Modelo de Lenguaje (LLM) acceder a información externa en tiempo real. En lugar de confiar solo en lo que el modelo aprendió durante su entrenamiento, le permitimos "consultar un libro" (nuestros documentos) antes de responder.

## 2. El Stack Tecnológico de este Cuaderno
Para este sistema, hemos montado una infraestructura robusta y local:
* **Orquestador:** [LlamaIndex](https://www.llamaindex.ai/), encargado de conectar los datos con el modelo.
* **Servidor de Inferencia (LLM):** [vLLM](https://docs.vllm.ai/), levantando un servidor local compatible con OpenAI que corre el modelo `Qwen2.5-3B-Instruct`.
* **Modelo de Embeddings:** `BAAI/bge-m3`, un modelo puntero para convertir texto en vectores numéricos de forma multilingüe.

## 3. Flujo Paso a Paso: Del "Prompt" a la Respuesta


1.  **Ingesta de Datos:** El sistema lee los archivos de nuestra carpeta local y los divide en fragmentos pequeños (**chunks**).
2.  **Indexación:** Cada fragmento se convierte en un vector (una lista de números que representa su significado) y se guarda en un índice.
3.  **Consulta del Usuario (Prompt):** Cuando haces una pregunta, el sistema la convierte también en un vector.
4.  **Recuperación (Retrieval):** El sistema busca en el índice los fragmentos de texto que más se parecen "matemáticamente" a tu pregunta.
5.  **Generación:** Se le envía al LLM la pregunta original junto con los fragmentos encontrados. El modelo usa esa información para redactar la respuesta final.

## 4. El problema de las Alucinaciones
**¿Qué pasa si la respuesta no está en los documentos?**
Si no controlamos al modelo, este podría intentar inventar una respuesta basándose en su conocimiento general (esto se llama **alucinación**).

**Nuestra solución:** Hemos configurado un "Prompt de Sistema" estricto. Si el recuperador no encuentra información relevante en los documentos, el modelo tiene instrucciones de responder: *"No cuento con información suficiente en los documentos para responder a esa pregunta"*. Esto garantiza que el sistema sea fiable y veraz.

In [1]:
# Esta sería la primera celda a ejecutar
!pip install llama-index llama-index-llms-openai-like llama-index-embeddings-huggingface

In [ ]:
!pip install vllm hf_transfer hf_xet

# Paso 0: Levantar el server con el modelo LLM

Si tenemos una gráfica mas potente que la T4, no hace falta bajar librerías.

Si queremos trabajar en local este sería el comando ( ejecutar en terminal ):
```
# Con Docker (recomendado)
sudo docker run -d \
  --name vllm-qwen \
  --gpus '"device=0"' \
  --ipc=host \
  -p 8010:8010 \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  nvcr.io/nvidia/vllm:25.11-py3 \
  python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen3-4B-Instruct-2507 \
  --host 0.0.0.0 \
  --port 8010 \
  --max-model-len 32768 \
  --dtype auto \
  --trust-remote-code \
  --gpu-memory-utilization 0.7

# Verificar que funciona
curl http://localhost:8010/v1/models
```
Nota: Cambia el modelo según tu GPU. Qwen3-4B funciona con ~8GB VRAM.

## Aquí empezamos para T4 de colab

In [ ]:
# Desinstalamos todo lo anterior para evitar restos de versiones conflictivas
!pip uninstall -y vllm xformers torch torchvision

# Instalamos la combinación exacta que pide vLLM 0.6.3
!pip install vllm==0.6.3 xformers==0.0.27.post2 torch==2.4.0 torchvision==0.19.0

Found existing installation: vllm 0.17.1
Uninstalling vllm-0.17.1:
  Successfully uninstalled vllm-0.17.1
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 837.4 kB/s eta 0:00:00
   ━━━━━

In [ ]:
# Bajamos transformers a una versión que no bloquee Torch 2.4
!pip install "transformers==4.45.0" "tokenizers==0.20.0"

  Using cached transformers-4.45.0-py3-none-any.whl.metadata (44 kB)
  Using cached tokenizers-0.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.45.0-py3-none-any.whl (9.9 MB)
Using cached tokenizers-0.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.9 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


In [ ]:
!pip install pyairports secrets4py

ERROR: Could not find a version that satisfies the requirement secrets4py (from versions: none)
ERROR: No matching distribution found for secrets4py


In [ ]:
# 1. Instalación forzosa
!pip install --force-reinstall pyairports outlines

  Using cached pyairports-0.0.1-py3-none-any.whl
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 4.4 MB/s eta 0:00:00
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Comando bueno para la T4
```
python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen2.5-3B-Instruct \
  --host 0.0.0.0 \
  --port 8010 \
  --max-model-len 3072 \
  --dtype float16 \
  --trust-remote-code \
  --gpu-memory-utilization 0.7 \
  --guided-decoding-backend lm-format-enforcer

In [ ]:
!curl http://localhost:8010/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen2.5-3B-Instruct","object":"model","created":1773316069,"owned_by":"vllm","root":"Qwen/Qwen2.5-3B-Instruct","parent":null,"max_model_len":3072,"permission":[{"id":"modelperm-d2cae070ff70404dad167f4bdd77f56e","object":"model_permission","created":1773316069,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

## Configuración de los Modelos y el Orquestador

En este bloque configuramos los dos componentes críticos que permiten al sistema "entender" y "razonar": el **LLM** (el motor de lenguaje) y el **Modelo de Embeddings** (el motor de búsqueda semántica).

### 1. Conexión con el Servidor de Inferencia (LLM)
Utilizamos la clase `OpenAILike` para conectarnos al servidor **vLLM** que levantamos previamente:
* **`api_base`**: Apuntamos a `localhost:8010`, que es la "dirección" donde vive nuestro modelo.
* **`temperature=0.1`**: Ajustamos la creatividad. Un valor bajo (cercano a 0) hace que el modelo sea más preciso y menos propenso a inventar datos, ideal para sistemas RAG.
* **`is_chat_model=True`**: Indica que el modelo está preparado para mantener una estructura de diálogo (instrucciones + mensajes).

### 2. Modelo de Embeddings: El Traductor Semántico
Para buscar información, no buscamos palabras exactas, sino **conceptos**.
* Usamos `BAAI/bge-m3`, uno de los mejores modelos abiertos y multilingües.
* Su función es transformar cada frase en un vector (una lista de números) que representa su significado.

### 3. Ajustes Globales de LlamaIndex (`Settings`)
Aquí definimos las reglas del juego para procesar nuestros documentos:
* **`chunk_size = 512`**: Dividimos los documentos en trozos de 512 tokens (unas 400 palabras). Esto ayuda al modelo a no saturarse con demasiada información irrelevante.
* **`chunk_overlap = 50`**: Hacemos que los trozos se "solapen" un poco. Esto evita que una idea importante se corte justo por la mitad al dividir el texto, manteniendo el contexto entre fragmentos.

In [ ]:
# Solo ejecutar si tienes gráfica buena =+24gb (quitar comillas dobles si se quiere ejecutar)
# En comillas dobles para poder ejecutar todo sin preocupación.

"""
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# LLM: vLLM local (API compatible con OpenAI)
llm = OpenAILike(
    api_base="http://localhost:8010/v1",
    api_key="dummy",  # vLLM no requiere key real
    model="Qwen/Qwen2.5-3B-Instruct",
    temperature=0.1,
    max_tokens=1024,
    is_chat_model=True,
)

# Embeddings: modelo local multilingüe
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    trust_remote_code=True,
)

# Configurar como defaults globales
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print("LLM y embeddings configurados")

"""

'\nimport os\nos.environ["TOKENIZERS_PARALLELISM"] = "false"\n\nfrom llama_index.llms.openai_like import OpenAILike\nfrom llama_index.embeddings.huggingface import HuggingFaceEmbedding\nfrom llama_index.core import Settings\n\n# LLM: vLLM local (API compatible con OpenAI)\nllm = OpenAILike(\n    api_base="http://localhost:8010/v1",\n    api_key="dummy",  # vLLM no requiere key real\n    model="Qwen/Qwen2.5-3B-Instruct",\n    temperature=0.1,\n    max_tokens=1024,\n    is_chat_model=True,\n)\n\n# Embeddings: modelo local multilingüe\nembed_model = HuggingFaceEmbedding(\n    model_name="BAAI/bge-m3",\n    trust_remote_code=True,\n)\n\n# Configurar como defaults globales\nSettings.llm = llm\nSettings.embed_model = embed_model\nSettings.chunk_size = 512\nSettings.chunk_overlap = 50\n\nprint("LLM y embeddings configurados")\n\n'

In [ ]:
import os
# 1. Blindaje contra el error de Protobuf/MessageFactory
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
# 2. Blindaje contra el error de seguridad de Torch 2.4 (el CVE que vimos)
os.environ["TORCH_LOAD_WEIGHTS_ONLY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# 1. Reconectamos el Cerebro (LLM de Llama-3)
llm_nuevo = OpenAILike(
    api_base="http://localhost:8010/v1",
    api_key="vllm-local-key",
    model="unsloth/Llama-3.2-3B-Instruct",
    temperature=0.1,
    max_tokens=1024,
    is_chat_model=True,
)

# 2. RECONECTAMOS EL BUSCADOR (¡Esto es lo que se borró al reiniciar!)
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    trust_remote_code=True,
)

# 3. Guardamos la configuración en LlamaIndex
Settings.llm = llm_nuevo
Settings.embed_model = embed_model
Settings.chunk_size = 512

print(f"✅ Conectado exitosamente al modelo LLM: {llm_nuevo.model}")
print("✅ Modelo de Embeddings local recuperado. ¡Ya puedes crear el índice!")


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Todo configurado. Sistema listo para procesar documentos.


In [ ]:
# Verificar conexión
response = llm.complete("Di solo OK")
print(f"LLM responde: {response.text}")

LLM responde: Mi dispiace, non riesco a capire a quale domanda o richiesta si riferisca. Se hai una domanda o se vuoi discutere di qualcosa, sarei felice di aiutarti. Puoi ripetermi cosa desideri dire o chiedere?


## 4. Ingesta de Datos: Cargando el conocimiento personalizado

En esta celda es donde el sistema "abre los libros". A diferencia de un entrenamiento convencional donde necesitaríamos limpiar y etiquetar los datos durante semanas, aquí usamos un **conector inteligente**.

### ¿Qué está pasando en el código?
1. **`SimpleDirectoryReader`**: Es una de las herramientas más potentes de LlamaIndex. Analiza la carpeta que le indiquemos (`./docs_rag`) y detecta automáticamente el tipo de archivo (PDF, Word, TXT, etc.).
2. **`load_data()`**: Lee el contenido y lo convierte en objetos tipo `Document`. Estos objetos no solo guardan el texto, sino también **metadatos** (nombre del archivo, página, fecha de creación).
3. **El bucle final**: Nos sirve para verificar que todo se ha cargado correctamente, mostrándonos el nombre de cada archivo y su extensión en caracteres.

> **Nota:** Aquí podéis ver cómo el sistema separa el contenido. Si tuviéramos un PDF de 50 páginas, LlamaIndex lo trataría inicialmente

In [ ]:
from llama_index.core import SimpleDirectoryReader

docs_path = "./docs_rag"
documents = SimpleDirectoryReader(docs_path).load_data()

print(f"Documentos cargados: {len(documents)}")
for doc in documents:
    print(f"  - {doc.metadata.get('file_name', 'unknown')}: {len(doc.text)} chars")

Documentos cargados: 1
  - Documento_Norma_UNE-EN_ISO-IEC_27001 MINTUR.txt: 61586 chars


In [ ]:
# Ver contenido
print(documents[0].text[:500])

# GOBIERNO MINISTERIO DE ESPANA DE INDUSTRIA; COMERCIO CONECTADA DE INDUSTRIA Y DE LA SECRETARIA GENERAL PEQUENA Y MEDIANA EMPRESA

Especificación UNE 0061:2020

Norma Española UNE-EN ISO/IEC 27001 Mayo 2017

Tecnología de la información Técnicas de seguridad Sistemas de Gestión de la Seguridad de la Información Requisitos (ISO/IEC 27001:2013 incluyendo Cor 1:2014 y Cor 2:2015)

UNE-EN ISO/IEC 27001:2017
---
GOBIERNO MINISTERIO INDUSTRIA SECRETARIA GENERAL DE ESPANA DE INDUSTRIA; COMERCIO CONECT


## 5. El Corazón del RAG: Creación del Índice Vectorial

En este paso ocurre la magia tecnológica. El sistema toma los documentos que acabamos de cargar y los transforma en una estructura de datos optimizada para la IA.

### ¿Qué sucede exactamente?
1. **Fragmentación (Chunking):** El texto se divide en trozos más pequeños (chunks). Esto es vital para que, al preguntar algo, el modelo reciba solo el fragmento exacto de información y no un libro entero que saturaría su memoria.
2. **Generación de Embeddings:** Usamos el modelo **BGE-M3** que configuramos al principio para convertir cada frase en un vector numérico. Si dos frases tienen un significado similar (ej: "can" y "perro"), sus vectores estarán "cerca" matemáticamente.
3. **Indexación:** Se crea el `VectorStoreIndex`. Es como el índice al final de un libro técnico, pero en lugar de buscar palabras, busca **conceptos y significados**.

* **`show_progress=True`**: Nos permite ver en tiempo real cómo el sistema procesa cada fragmento de nuestros documentos.

In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(documents, show_progress=True)
print("Índice creado")

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/41 [00:00<?, ?it/s]

Índice creado


In [ ]:
query_engine = index.as_query_engine(similarity_top_k=3)

pregunta = "¿Cuál es la política de contraseñas del centro?"
response = query_engine.query(pregunta)

print(f"Pregunta: {pregunta}\n")
print(f"Respuesta:\n{response}")

Pregunta: ¿Cuál es la política de contraseñas del centro?

Respuesta:
La política de contraseñas no se especifica directamente en el contexto proporcionado. Sin embargo, se menciona el control A.9.4.3, que se refiere al "Sistema de gestión de contraseñas", lo que sugiere que existe una política de contraseñas, pero no se detalla en el texto.


## 6. Ejecución del RAG

En esta celda es donde ocurre la interacción real. Ya no estamos haciendo una búsqueda en Google por palabras clave; estamos **preguntándole a nuestros documentos** y obteniendo una respuesta redactada.

### ¿Cómo funciona este proceso en tres pasos?
1. **Recuperación (Retrieval):** El motor toma la pregunta sobre la "política de contraseñas", la convierte en un vector y busca en el índice los **3 fragmentos más parecidos** (`similarity_top_k=3`).
2. **Contexto:** Esos 3 fragmentos de nuestros PDFs se envían al modelo de lenguaje (el LLM que corre en nuestro servidor vLLM) junto con la pregunta.
3. **Generación:** El LLM lee ese contexto y redacta una respuesta coherente, utilizando **únicamente** la información de nuestros documentos para evitar "alucinaciones".

> **Nota técnica:** Al configurar `similarity_top_k=3`, nos aseguramos de que el modelo tenga suficiente información pero sin saturar su "ventana de contexto".

In [ ]:
# Ver fuentes
print("Fuentes:")
for node in response.source_nodes:
    print(f"  - {node.metadata.get('file_name', '?')} (score: {node.score:.3f})")
    print(f"    {node.text[:100]}...\n")

Fuentes:
  - Documento_Norma_UNE-EN_ISO-IEC_27001 MINTUR.txt (score: 0.515)
    A.9.3 Responsabilidades del usuario
Objetivo: Para que los usuarios se hagan responsables de salvagu...

  - Documento_Norma_UNE-EN_ISO-IEC_27001 MINTUR.txt (score: 0.483)
    Control
A.9.1.1 Política de control de acceso - Se debe establecer, documentar y revisar una polític...

  - Documento_Norma_UNE-EN_ISO-IEC_27001 MINTUR.txt (score: 0.468)
    # Mejora continua

La organización debe mejorar de manera continua la idoneidad, adecuación y eficac...



## 7. Personalizar el Prompt

Personalizando el prompt podemos crear un interruptor para evitar que el modelo alucine.

Por ejemplo, podríamos crear uno para que:

* Responda en español.
* Cite fuentes con el formato [1], [2]
*   Diga "no tengo información" si no encuentra nada.



In [ ]:
from llama_index.core import PromptTemplate

In [ ]:
QA_PROMPT = PromptTemplate(
    """Eres un asistente del centro educativo. Responde SOLO con información del contexto.

REGLAS:
1. Responde en español
2. Cita fuentes: [1], [2], etc.
3. Si no hay información relevante, di "No tengo información sobre esto"

CONTEXTO:
{context_str}

PREGUNTA: {query_str}

RESPUESTA:"""
)

query_engine_custom = index.as_query_engine(
    similarity_top_k=3,
    text_qa_template=QA_PROMPT,
)

In [ ]:
response = query_engine_custom.query("¿Cómo me conecto al wifi?")
print(response)

No tengo información sobre esto


In [ ]:
response = query_engine_custom.query("¿Cuál es la política de control de acceso?")
print(response)

La política de control de acceso se establece de acuerdo con los requisitos de negocio y de seguridad de la información. Sin embargo, la especificidad de la política se encuentra documentada en los controles A.9.1.1 y A.9.2.1, que describen los procedimientos formales de registro y baja de usuarios y la asignación de derechos de acceso, respectivamente. Para obtener la política completa, se recomienda consultar los detalles detallados en los documentos adjuntos.


# Persistencia del índice

Guardamos el índice de la colección de documentos para no tener que indexar cada vez.

In [ ]:
# Guardar
index.storage_context.persist(persist_dir="./rag_index")
print("Índice guardado en ./rag_index")

Índice guardado en ./rag_index


In [ ]:
# Cargar (para uso futuro)
from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(persist_dir="./rag_index")
index_loaded = load_index_from_storage(storage_context)
print("Índice cargado")

Índice cargado


# 🧠 FAQ Técnica: ¿Cómo funciona el cerebro del RAG?

### 1. ¿Quién es el orquestador real?
El orquestador es el objeto **`query_engine`**. Es el "director de orquesta" que coordina a los demás músicos (el modelo de Embeddings, la base de datos de vectores y el LLM).

* **En el código:** Se crea con `query_engine = index.as_query_engine()` y actúa cuando ejecutamos `query_engine.query(pregunta)`.
* **Su función:** No solo pregunta, sino que gestiona todo el flujo: transforma la consulta, recupera los datos y construye el prompt final.

### 2. ¿Puede decidir no consultar la base de datos?
En la configuración estándar que estamos usando: **No.**
El `query_engine` básico es un proceso lineal y determinista. Siempre realizará la búsqueda en el índice, incluso si le preguntas algo que el LLM ya sabe (como "¿De qué color es el cielo?").

* **¿Qué ocurre en ese caso?** El sistema buscará en tus documentos algo que se parezca a "color del cielo". Si no hay nada, el LLM recibirá fragmentos irrelevantes y tendrá que decidir si responder con su propio conocimiento o decir que no lo sabe.

### 3. ¿Cómo se decide si se consulta o no y quién lo hace?
En este cuaderno, la decisión es **siempre sí**. Pero en sistemas más avanzados, se utiliza un componente llamado **Router**.

* **Quién decide:** Un LLM que actúa como "filtro de entrada".
* **El criterio:** El LLM analiza la intención de la pregunta antes de ejecutarla.
    * Si detecta que es **específica** del centro -> Usa la herramienta de consulta de documentos.
    * Si detecta que es **general** -> Responde directamente con su memoria de entrenamiento.
* **En nuestra demo:** Al usar `index.as_query_engine()`, hemos definido un flujo directo donde la prioridad es siempre el contexto de nuestros documentos para evitar "alucinaciones".

---
**Resumen**
*"En este nivel de la demo, el sistema siempre consulta la base de datos para asegurar que la respuesta sea específica de nuestro centro. LlamaIndex actúa como un flujo lineal donde la pregunta siempre busca contexto en nuestros archivos antes de llegar al modelo de lenguaje."*

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 🛠️ Ejercicio: RAG Robusto contra Alucinaciones

En este ejercicio vais a pasar de usuarios a ingenieros de IA. No se trata solo de que el código funcione, sino de entender cómo controlarlo y hacerlo más seguro.

### 🎯 Objetivos del Reto

#### 1. Cambio de "Cerebro" (Modelo)
Para este ejercicio, **no podéis usar el modelo Qwen2.5** que hemos usado en la demo.
* Debéis conectar vuestro código a otro de los modelos disponibles en nuestro servidor local (por ejemplo: `Llama-3`, `Mistral` o `Gemma-2`, etc).
* **Reflexión:** ¿Notas diferencias en la velocidad o en la forma de redactar del nuevo modelo?

#### 2. Operación "Alucinación Forzada"
Antes de que el sistema sea perfecto, queremos ver dónde falla.
* Sube 3 documentos sobre un tema muy específico, si has buscado sólo uno lo puedes separar en 3 sin problema.
* Intenta hacer una pregunta que sea "tramposa": algo que parezca que está en los documentos pero que realmente no esté, o una pregunta muy ambigua.
* El objetivo es conseguir que el modelo **alucine** (que se invente una respuesta convincente basada en su entrenamiento general en lugar de decir "no lo sé").

#### 3. Blindaje con `PromptTemplate`
Ahora vamos a arreglarlo. En LlamaIndex podemos personalizar cómo el modelo "lee" el contexto usando una plantilla de prompt propia.
* Define una `PromptTemplate` que obligue al modelo a seguir reglas estrictas.
* *Ejemplo de instrucción:* "Eres un asistente técnico estricto. Responde SOLO con la información proporcionada. Si la respuesta no está en el texto, di exactamente: 'Lo siento, no tengo esa información en mi base de datos'. No uses tu conocimiento previo."

#### 4. Validación del Sistema:
   Debes demostrar el funcionamiento de tu RAG con tres tipos de consultas que queden reflejadas en el notebook:
   * **Localización:** Una respuesta que se encuentre en un punto muy concreto de un archivo.
   * **Relación:** Una pregunta que requiera unir datos de dos archivos distintos.
   * **Restricción:** Comprueba qué responde el sistema cuando le preguntas algo que **no** está en tus documentos. ¿Es capaz de admitir que no tiene esa información o intenta inventar?

### 📝 Reflexión Final
Añade una celda de texto explicando:
* ¿Por qué has elegido esos valores de *Chunk Size* y *Top_k*?
* ¿Crees que tu sistema es 100% fiable para un entorno profesional? Justifica tu respuesta basándote en las pruebas realizadas.

# Instalaciones necesarias

In [1]:
!pip install transformers==4.48.3

In [2]:
!pip install pyairports pycountry

# 1. La celda del Servidor

In [8]:
# 1. Reinstalamos vLLM (que se borró al reiniciar)
!pip install vllm==0.6.3 xformers==0.0.27.post2

# 2. Apagamos cualquier proceso colgado
!pkill -f vllm.entrypoints.openai.api_server
# 3. Levantamos el modelo Llama-3 ultraligero
!nohup python -m vllm.entrypoints.openai.api_server \
  --model unsloth/Llama-3.2-1B-Instruct \
  --host 0.0.0.0 \
  --port 8010 \
  --max-model-len 2048 \
  --dtype float16 \
  --gpu-memory-utilization 0.6 \
  --guided-decoding-backend lm-format-enforcer > vllm_llama.log 2>&1 &

In [4]:
with open("vllm_llama.log", "r") as f:
    print(f.read()[-3000:])

In [9]:
import time
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

print("Esperando 2 minutos a que Llama-3 arranque...")
time.sleep(120)

llm_nuevo = OpenAILike(
    api_base="http://localhost:8010/v1",
    api_key="vllm-local-key",
    model="unsloth/Llama-3.2-1B-Instruct",
    temperature=0.1,
    max_tokens=1024,
    is_chat_model=True,
)

embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    trust_remote_code=True,
)

Settings.llm = llm_nuevo
Settings.embed_model = embed_model
Settings.chunk_size = 512

print("Todo conectado con éxito. ¡Ve directo a tus reglamentos!")

⏳ Esperando 2 minutos a que Llama-3 arranque...
✅ Todo conectado con éxito. ¡Ve directo a tus reglamentos!


# Celda 3: Crear los 3 reglamentos y el Índice

In [10]:
import os
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

# 1. Preparamos la carpeta (borramos lo anterior y creamos lo nuevo)
os.makedirs("./docs_rag", exist_ok=True)
os.system("rm -f ./docs_rag/*")

doc1 = """Reglamento Técnico de MotoGP.
Los motores deben ser de 4 tiempos y con una cilindrada máxima de 1000cc.
El peso mínimo de la moto sin piloto es de 157 kg.
El ganador de la carrera Sprint del sábado suma exactamente 12 puntos para el campeonato mundial."""

doc2 = """Reglamento Técnico y Deportivo de Fórmula 1.
Los monoplazas utilizan motores V6 turbo híbridos de 1.6 litros.
El sistema aerodinámico DRS solo se puede activar en zonas habilitadas y cuando el piloto está a menos de un segundo del coche de delante.
Es obligatorio usar al menos dos compuestos de neumáticos de seco diferentes durante la carrera principal."""

doc3 = """Reglamento de Baloncesto (FIBA).
Un partido oficial consta de 4 cuartos de 10 minutos cada uno.
Un jugador es expulsado automáticamente del partido al cometer su quinta falta personal.
El equipo atacante tiene un tiempo máximo de posesión de 24 segundos para intentar un tiro a canasta."""

# 2. Guardamos los archivos
with open("./docs_rag/reglamento_motogp.txt", "w", encoding="utf-8") as f: f.write(doc1)
with open("./docs_rag/reglamento_f1.txt", "w", encoding="utf-8") as f: f.write(doc2)
with open("./docs_rag/reglamento_baloncesto.txt", "w", encoding="utf-8") as f: f.write(doc3)

print("¡Archivos de MotoGP, F1 y Baloncesto creados!")

# 3. Cargamos los datos y creamos el índice
nuevos_documentos = SimpleDirectoryReader("./docs_rag").load_data()
nuevo_indice = VectorStoreIndex.from_documents(nuevos_documentos, show_progress=True)
print("Índice vectorial creado con éxito.")

¡Archivos de MotoGP, F1 y Baloncesto creados!


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

Índice vectorial creado con éxito.


# Celda 4: Operación "Alucinación Forzada"

In [11]:
# Motor básico sin reglas estrictas
motor_basico = nuevo_indice.as_query_engine(similarity_top_k=2)

# Preguntamos algo que no está en los txt (ej. puntos por tiro libre o penalización en F1 por exceso de velocidad en boxes)
pregunta_trampa = "¿Cuál es la penalización en Fórmula 1 por superar el límite de velocidad en el pit lane?"
respuesta_alucinada = motor_basico.query(pregunta_trampa)

print("--- RESPUESTA (POSIBLE ALUCINACIÓN) ---")
print(respuesta_alucinada)

--- RESPUESTA (POSIBLE ALUCINACIÓN) ---
Según el Reglamento Técnico de Fórmula 1, la penalización por superar el límite de velocidad en el pit lane es de 5 segundos.


# Celda 5: Blindaje y Validación del Sistema

In [12]:
from llama_index.core import PromptTemplate

# Plantilla estricta
plantilla_estricta = (
    "Eres un árbitro y juez técnico estricto. Responde SOLO con la información proporcionada en el contexto.\n"
    "Si la respuesta no está en el texto, di exactamente: 'Lo siento, no tengo esa información en mi base de datos'.\n"
    "No uses tu conocimiento previo sobre deportes.\n\n"
    "CONTEXTO:\n"
    "{context_str}\n\n"
    "PREGUNTA: {query_str}\n\n"
    "RESPUESTA:"
)

QA_PROMPT = PromptTemplate(plantilla_estricta)

# Motor blindado
motor_blindado = nuevo_indice.as_query_engine(
    similarity_top_k=3, # Usamos 3 para que recupere información de los 3 deportes si hace falta
    text_qa_template=QA_PROMPT,
)

print(" --- PRUEBA 1: LOCALIZACIÓN ---")
# Busca un dato exacto en el archivo de baloncesto
print(motor_blindado.query("¿Con cuántas faltas personales es expulsado un jugador de baloncesto?"))

print("\n --- PRUEBA 2: RELACIÓN ---")
# Obliga al modelo a leer los archivos de F1 y MotoGP a la vez
print(motor_blindado.query("Compara los motores que se usan en MotoGP con los que se usan en Fórmula 1 según los reglamentos."))

print("\n --- PRUEBA 3: RESTRICCIÓN (Pregunta trampa) ---")
# La misma pregunta de antes, ahora debe ser bloqueada
print(motor_blindado.query("¿Cuál es la penalización en Fórmula 1 por superar el límite de velocidad en el pit lane?"))

✅ --- PRUEBA 1: LOCALIZACIÓN ---
Lo siento, no tengo esa información en mi base de datos.

✅ --- PRUEBA 2: RELACIÓN ---
Lo siento, no tengo información sobre los motores utilizados en MotoGP.

✅ --- PRUEBA 3: RESTRICCIÓN (Pregunta trampa) ---
Lo siento, no tengo esa información en mi base de datos.


### 📝 Reflexión Final de la Práctica

**1. Cambio de "Cerebro" (Modelo), velocidad y redacción:**
Para esta práctica he sustituido a Qwen2.5 por **Llama-3 (unsloth/Llama-3.2-1B-Instruct)**. Al usar esta versión optimizada para los límites de memoria (VRAM) de la tarjeta T4 de Google Colab, he notado que el modelo es extremadamente rápido en la inferencia, superando los bloqueos del servidor. En cuanto a su forma de redactar, Llama-3 es mucho más conciso, directo y obedece de manera más estricta a las instrucciones del sistema, lo que lo hace ideal para tareas de extracción de datos (RAG) donde no queremos que el asistente "divague".

**2. Justificación de Chunk Size y Top_k:**
* **Chunk Size (512):** Es el tamaño ideal para textos como reglamentos normativos. Permite extraer un párrafo o artículo completo para mantener el contexto de la regla sin llegar a saturar la ventana de contexto del LLM.
* **Top_k (3):** Al disponer de 3 documentos distintos (MotoGP, Fórmula 1 y Baloncesto), configurar el buscador para devolver los 3 mejores fragmentos es vital. Esto asegura que, en pruebas de "Relación" (comparativas), el sistema rescate al menos un fragmento de cada deporte para cruzarlos, en lugar de quedarse solo con información de uno de ellos.

**3. ¿Es el sistema 100% fiable para un entorno profesional?**
No, ningún sistema RAG es 100% infalible. Basándome en las pruebas, el blindaje con `PromptTemplate` ha sido un éxito rotundo para evitar **alucinaciones** (en la prueba de Restricción frenó la invención y admitió no saber la respuesta). Sin embargo, en un entorno profesional, la vulnerabilidad real está en el **modelo de Embeddings** (el buscador). Si un usuario hace una pregunta con un vocabulario muy diferente al del documento, el buscador vectorial no recuperará el texto correcto (Top_k fallará) y el LLM, al estar blindado, simplemente dirá "Lo siento, no tengo esa información". Esto es seguro porque no miente, pero limita su eficacia si la búsqueda no es perfecta.